# 02 - Segmentation ablation: rembg vs SAM

Mask quality caps the whole pipeline: stage 2 decides what to preserve from the mask alone.
Two tools with opposite philosophies are compared - **rembg** (u2net, single-purpose salient-object removal, no knobs) and **SAM ViT-B** (general-purpose, promptable with boxes and points).

Test set: 5 products, 4 categories, same table and lighting - matte (mug, book), glossy/thin detail (fork), transparent (patterned glass), low contrast (black headphones on dark wood).

**Findings**
- Matte class: the two methods are interchangeable (IoU ~0.99).
- Thin detail: SAM resolves the fork tines, rembg rounds them off.
- Transparent: IoU 0.99, but both produce a *solid* mask - correct for inpainting (the optics behind glass cannot be replaced anyway), wrong for compositing.
- Low contrast: IoU 0.23. SAM's box prompt grabbed only the fabric headband; rembg covered the whole object but filled a genuine hole between the band and the ear cups.
- IoU measures *agreement*, not correctness, and is sensitive to object size (the fork's 0.93 comes from a 2.4% area).

**Decision:** rembg as the default; SAM where fine detail matters; the low-contrast class is a documented limitation.

Extra dependencies: `pip install segment-anything` and the ViT-B checkpoint in `models/sam_vit_b.pth`.

In [ ]:
import os, sys
sys.path.insert(0, "../src")
os.chdir("..")
print("working dir:", os.getcwd())

## 1. rembg (u2net)

`only_mask=True` returns the mask itself rather than a cut-out. Its character: soft, slightly dilated edges (1-3 px wider than the product).

In [ ]:
from rembg import remove, new_session
from PIL import Image

session = new_session("u2net")
image = Image.open("inputs/testset/01_matte_mug.jpeg").resize((512, 640))

mask = remove(image, session=session, only_mask=True)
mask.save("outputs/mask_rembg.png")
print("mask_rembg.png ready")

## 2. SAM ViT-B with a box prompt

`set_image` is the expensive step (image encoding); `predict` is cheap, so many prompts can be tried on one image.
Box format is **[x1, y1, x2, y2]** (top-left then bottom-right), with y increasing downwards.

In [ ]:
import numpy as np
import torch
from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry["vit_b"](checkpoint="models/sam_vit_b.pth").to("cuda")
predictor = SamPredictor(sam)
predictor.set_image(np.array(image))

masks, scores, _ = predictor.predict(
    box=np.array([90, 150, 440, 500]), multimask_output=False)

Image.fromarray((masks[0] * 255).astype(np.uint8)).save("outputs/mask_sam.png")
print(f"mask_sam.png ready - confidence {scores[0]:.3f}")

## 3. IoU and difference map

Red = SAM only, blue = rembg only, white = both. On the mug: IoU 0.9874, with a thin blue band showing rembg's wider edge.

In [ ]:
from metrics import mask_iou, mask_diff_map

mask_r = Image.open("outputs/mask_rembg.png")
mask_s = Image.open("outputs/mask_sam.png")

print(f"rembg vs SAM IoU: {mask_iou(mask_r, mask_s):.4f}")
mask_diff_map(mask_r, mask_s).save("outputs/mask_diff.png")

## 4. Test set overview

The grid overlay makes it easy to read box coordinates off each image by eye.

In [ ]:
import glob
import matplotlib.pyplot as plt

files = sorted(glob.glob("inputs/testset/*.jpeg"))
fig, axes = plt.subplots(1, len(files), figsize=(20, 6))
for ax, path in zip(axes, files):
    ax.imshow(Image.open(path).resize((512, 640)))
    ax.set_title(os.path.basename(path), fontsize=8)
    ax.set_xticks(range(0, 513, 64))
    ax.set_yticks(range(0, 641, 64))
    ax.grid(True, color="red", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
BOXES = {
    "01_matte_mug":            [90, 150, 440, 500],
    "02_matte_book":           [64, 192, 430, 420],
    "03_glossy_fork":          [50, 320, 448, 420],
    "04_transparent_glass":    [128, 192, 374, 500],
    "05_lowcontrast_headphones": [16, 192, 500, 512],
}

## 5. Batch ablation

Models are loaded once, outside the loop. The area column is a cheap sanity check: on the headphones rembg reports 28.8% and SAM 6.7%, which flags a missed object long before looking at IoU.

In [ ]:
import csv

os.makedirs("outputs/masks", exist_ok=True)
files = sorted(glob.glob("inputs/testset/*.jpeg"))
assert files, "test set is empty"

rows = []
for path in files:
    name = os.path.basename(path).replace(".jpeg", "")
    img = Image.open(path).resize((512, 640))

    m_rembg = np.array(remove(img, session=session, only_mask=True))
    Image.fromarray(m_rembg).save(f"outputs/masks/rembg_{name}.png")

    predictor.set_image(np.array(img))
    masks, scores, _ = predictor.predict(
        box=np.array(BOXES[name]), multimask_output=False)
    m_sam = (masks[0] * 255).astype(np.uint8)
    Image.fromarray(m_sam).save(f"outputs/masks/sam_{name}.png")

    b_r, b_s = m_rembg > 127, m_sam > 127
    iou = np.logical_and(b_r, b_s).sum() / np.logical_or(b_r, b_s).sum()
    mask_diff_map(Image.fromarray(m_rembg), Image.fromarray(m_sam)).save(
        f"outputs/masks/diff_{name}.png")

    rows.append([name, name.split("_")[1], f"{iou:.4f}",
                 f"{b_r.mean()*100:.1f}", f"{b_s.mean()*100:.1f}", f"{scores[0]:.3f}"])
    print(f"{name}: IoU {iou:.4f} | rembg {b_r.mean()*100:.1f}% | "
          f"SAM {b_s.mean()*100:.1f}% | confidence {scores[0]:.3f}")

with open("outputs/ablation.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["product", "category", "iou", "area_rembg_pct",
                     "area_sam_pct", "sam_confidence"])
    writer.writerows(rows)
print("ablation.csv written")

## 6. Prompt tuning on the failure case

Adding two positive points on the ear cups pulls them into the mask, but the edges stay ragged: the prompt is a hyperparameter, yet neither method is sufficient alone in this class.

In [ ]:
name = "05_lowcontrast_headphones"
predictor.set_image(np.array(
    Image.open(f"inputs/testset/{name}.jpeg").resize((512, 640))))

masks, scores, _ = predictor.predict(
    box=np.array(BOXES[name]),
    point_coords=np.array([[230, 300], [380, 330]]),  # ear cup centres
    point_labels=np.array([1, 1]),                    # 1 = positive
    multimask_output=False)

tuned = (masks[0] * 255).astype(np.uint8)
Image.fromarray(tuned).save(f"outputs/masks/sam_tuned_{name}.png")
print(f"confidence: {scores[0]:.3f} | area: {(tuned > 127).mean()*100:.1f}%")